# Analyse statistique des résultats Transformer

Tests de Friedman, Nemenyi (post hoc), Wilcoxon et ANOVA.

**Statut :** Retenu (tableaux 15 et 16)

*Mémoire de maîtrise — Mireille Kafando, UQTR · févr.–mars 2025*

> Notebook de recherche conçu pour Google Colab. Les chemins `/content/drive/MyDrive/bug/...` sont à adapter (voir `data/README.md`). Les sorties ont été retirées.


In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import seaborn as sns
from scipy.stats import shapiro, ttest_rel, wilcoxon

# Création du dataframe avec les données du tableau
systemes = ["Ant", "Camel", "Ivy", "Jedit", "Log4j", "Lucene", "Poi", "Synapse", "Velocity", "Xalan", "Xerces"]
gmean_cv = [68.71, 65.5, 72.57, 60.8, 54.43, 64.44, 65.49, 60.38, 63.76, 62.96, 45.53]
gmean_csv = [69.16, 59, 74.68, 56.85, 54.92, 53.05, 68.27, 67.83, 54.4, 61.08, 60.17]

# Création du dataframe
data = pd.DataFrame({
    'systemes': systemes,
    'gmean_cv': gmean_cv,
    'gmean_csv': gmean_csv
})

# Affichage du dataframe
print("Données d'entrée:")
display(data)

# 1. Visualisation des données
plt.figure(figsize=(12, 5))

# Boxplot pour comparer les distributions
plt.subplot(1, 2, 1)
data_melted = pd.melt(data, id_vars='systemes', value_vars=['gmean_cv', 'gmean_csv'],
                     var_name='method', value_name='gmean')
sns.boxplot(x='method', y='gmean', data=data_melted)
plt.title("Distribution des valeurs Gmean par méthode")
plt.xlabel("Méthode")
plt.ylabel("Valeur Gmean (%)")

# 2. Test de normalité avec Shapiro-Wilk
print("\n--- Test de normalité (Shapiro-Wilk) ---")
shapiro_cv = shapiro(data['gmean_cv'])
shapiro_csv = shapiro(data['gmean_csv'])

print(f"Test de Shapiro-Wilk pour %Gmean cv: W={shapiro_cv.statistic:.4f}, p-value={shapiro_cv.pvalue:.4f}")
print(f"Test de Shapiro-Wilk pour %Gmean csv: W={shapiro_csv.statistic:.4f}, p-value={shapiro_csv.pvalue:.4f}")

# 3. QQ-plots pour vérifier visuellement la normalité
plt.subplot(1, 2, 2)
stats.probplot(data['gmean_cv'], dist="norm", plot=plt)
plt.title("QQ-Plot pour %Gmean cv")

plt.figure(figsize=(6, 5))
stats.probplot(data['gmean_csv'], dist="norm", plot=plt)
plt.title("QQ-Plot pour %Gmean csv")

plt.tight_layout()
plt.show()

# 4. En fonction des résultats de normalité, choisir le test approprié
alpha = 0.05  # Seuil de significativité

if shapiro_cv.pvalue > alpha and shapiro_csv.pvalue > alpha:
    print("\nLes deux distributions semblent suivre une loi normale. Application du test t apparié:")
    t_test_result = ttest_rel(data['gmean_cv'], data['gmean_csv'])
    print(f"Test t apparié: t={t_test_result.statistic:.4f}, p-value={t_test_result.pvalue:.4f}")
    if t_test_result.pvalue < alpha:
        print("Conclusion: Il existe une différence significative entre les deux groupes.")
    else:
        print("Conclusion: Il n'existe pas de différence significative entre les deux groupes.")
else:
    print("\nAu moins une des distributions ne suit pas une loi normale. Application du test de Wilcoxon:")
    wilcox_test_result = wilcoxon(data['gmean_cv'], data['gmean_csv'])
    print(f"Test de Wilcoxon pour échantillons appariés: statistic={wilcox_test_result.statistic}, p-value={wilcox_test_result.pvalue:.4f}")
    if wilcox_test_result.pvalue < alpha:
        print("Conclusion: Il existe une différence significative entre les deux groupes.")
    else:
        print("Conclusion: Il n'existe pas de différence significative entre les deux groupes.")

# 5. Calculer des statistiques descriptives
print("\n--- Statistiques descriptives ---")
print(f"Moyenne %Gmean cv: {np.mean(data['gmean_cv']):.2f}")
print(f"Écart-type %Gmean cv: {np.std(data['gmean_cv'], ddof=1):.2f}")
print(f"Moyenne %Gmean csv: {np.mean(data['gmean_csv']):.2f}")
print(f"Écart-type %Gmean csv: {np.std(data['gmean_csv'], ddof=1):.2f}")
print(f"Différence moyenne: {np.mean(data['gmean_cv'] - data['gmean_csv']):.2f}")

# 6. Visualiser les différences
diff = data['gmean_cv'] - data['gmean_csv']
plt.figure(figsize=(12, 6))
plt.bar(data['systemes'], diff, color=np.where(diff >= 0, 'steelblue', 'tomato'))
plt.axhline(y=0, linestyle='--', color='black')
plt.title("Différence entre %Gmean cv et %Gmean csv par système")
plt.xlabel("Systèmes")
plt.ylabel("Différence (cv - csv)")
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Afficher les différences système par système
print("\n--- Différences par système ---")
data['diff'] = data['gmean_cv'] - data['gmean_csv']
display(data[['systemes', 'gmean_cv', 'gmean_csv', 'diff']].sort_values(by='diff'))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import seaborn as sns
from scipy.stats import shapiro, friedmanchisquare, f_oneway
from statsmodels.stats.anova import AnovaRM

# Création du dataframe avec les données du tableau
systemes = ["Camel", "Ivy", "Jedit", "Log4j", "Lucene", "Synapse", "Xerces", "Ant", "Poi", "Velocity", "Xalan"]
gmean_cv = [65.5, 72.57, 60.8, 54.43, 64.44, 60.38, 45.53, 68.71, 65.49, 63.76, 62.96]
gmean_csvt = [59, 74.68, 56.85, 54.92, 53.05, 67.83, 60.17, 69.16, 68.27, 54.4, 61.08]
gmean_csvs4 = [60.48, 69.45, 57.54, 59.71, 58.43, 60.49, 55.93, 69.09, 66.12, 58.28, 64.71]

# Création du dataframe
data = pd.DataFrame({
    'systemes': systemes,
    'gmean_cv': gmean_cv,
    'gmean_csvt': gmean_csvt,
    'gmean_csvs4': gmean_csvs4
})

# Affichage du dataframe
print("Données d'entrée:")
display(data)

# 1. Visualisation des données
plt.figure(figsize=(14, 6))

# Boxplot pour comparer les distributions
plt.subplot(1, 2, 1)
data_melted = pd.melt(data, id_vars='systemes',
                     value_vars=['gmean_cv', 'gmean_csvt', 'gmean_csvs4'],
                     var_name='method', value_name='gmean')
sns.boxplot(x='method', y='gmean', data=data_melted)
plt.title("Distribution des valeurs Gmean par méthode")
plt.xlabel("Méthode")
plt.ylabel("Valeur Gmean (%)")

# 2. Test de normalité avec Shapiro-Wilk
print("\n--- Test de normalité (Shapiro-Wilk) ---")
shapiro_cv = shapiro(data['gmean_cv'])
shapiro_csvt = shapiro(data['gmean_csvt'])
shapiro_csvs4 = shapiro(data['gmean_csvs4'])

print(f"Test de Shapiro-Wilk pour %Gmean cv: W={shapiro_cv.statistic:.4f}, p-value={shapiro_cv.pvalue:.4f}")
print(f"Test de Shapiro-Wilk pour %Gmean csvt: W={shapiro_csvt.statistic:.4f}, p-value={shapiro_csvt.pvalue:.4f}")
print(f"Test de Shapiro-Wilk pour %Gmean csvs4: W={shapiro_csvs4.statistic:.4f}, p-value={shapiro_csvs4.pvalue:.4f}")

# 3. QQ-plots pour vérifier visuellement la normalité
plt.subplot(1, 2, 2)
stats.probplot(data['gmean_cv'], dist="norm", plot=plt)
plt.title("QQ-Plot pour %Gmean cv")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
stats.probplot(data['gmean_csvt'], dist="norm", plot=plt)
plt.title("QQ-Plot pour %Gmean csvt")

plt.subplot(1, 2, 2)
stats.probplot(data['gmean_csvs4'], dist="norm", plot=plt)
plt.title("QQ-Plot pour %Gmean csvs4")
plt.tight_layout()
plt.show()

# 4. En fonction des résultats de normalité, choisir le test approprié
alpha = 0.05  # Seuil de significativité
all_normal = (shapiro_cv.pvalue > alpha and
              shapiro_csvt.pvalue > alpha and
              shapiro_csvs4.pvalue > alpha)

# Préparation des données pour les tests
df_long = pd.melt(data, id_vars='systemes',
                 value_vars=['gmean_cv', 'gmean_csvt', 'gmean_csvs4'],
                 var_name='method', value_name='gmean')

# Initialiser les variables pour stocker les résultats des tests
pairwise_results = {}

if all_normal:
    print("\nLes trois distributions semblent suivre une loi normale. Application de l'ANOVA pour mesures répétées:")

    # ANOVA pour mesures répétées
    aov = AnovaRM(data=df_long, depvar='gmean', subject='systemes', within=['method']).fit()
    print(aov.summary())

    if aov.anova_table['Pr > F'].iloc[0] < alpha:
        print("Conclusion: Il existe au moins une différence significative entre les trois groupes.")

        # Post-hoc tests: Pairwise t-tests
        print("\n--- Tests post-hoc: Comparaisons pairées (t-test apparié) ---")

        # CV vs CSVT
        t_cv_csvt = stats.ttest_rel(data['gmean_cv'], data['gmean_csvt'])
        print(f"CV vs CSVT: t={t_cv_csvt.statistic:.4f}, p-value={t_cv_csvt.pvalue:.4f}")
        pairwise_results['CV_CSVT'] = {'test': 't-test', 'statistic': t_cv_csvt.statistic, 'pvalue': t_cv_csvt.pvalue}

        # CV vs CSVS4
        t_cv_csvs4 = stats.ttest_rel(data['gmean_cv'], data['gmean_csvs4'])
        print(f"CV vs CSVS4: t={t_cv_csvs4.statistic:.4f}, p-value={t_cv_csvs4.pvalue:.4f}")
        pairwise_results['CV_CSVS4'] = {'test': 't-test', 'statistic': t_cv_csvs4.statistic, 'pvalue': t_cv_csvs4.pvalue}

        # CSVT vs CSVS4
        t_csvt_csvs4 = stats.ttest_rel(data['gmean_csvt'], data['gmean_csvs4'])
        print(f"CSVT vs CSVS4: t={t_csvt_csvs4.statistic:.4f}, p-value={t_csvt_csvs4.pvalue:.4f}")
        pairwise_results['CSVT_CSVS4'] = {'test': 't-test', 'statistic': t_csvt_csvs4.statistic, 'pvalue': t_csvt_csvs4.pvalue}

    else:
        print("Conclusion: Il n'existe pas de différence significative entre les trois groupes.")

else:
    print("\nAu moins une des distributions ne suit pas une loi normale. Application du test de Friedman:")

    # Test de Friedman (test non paramétrique pour k échantillons appariés)
    friedman_test = friedmanchisquare(data['gmean_cv'], data['gmean_csvt'], data['gmean_csvs4'])
    print(f"Test de Friedman: chi2={friedman_test.statistic:.4f}, p-value={friedman_test.pvalue:.4f}")

    if friedman_test.pvalue < alpha:
        print("Conclusion: Il existe au moins une différence significative entre les trois groupes.")

        # Post-hoc tests: Wilcoxon signed-rank test
        print("\n--- Tests post-hoc: Comparaisons pairées (test de Wilcoxon) ---")

        # CV vs CSVT
        w_cv_csvt = stats.wilcoxon(data['gmean_cv'], data['gmean_csvt'])
        print(f"CV vs CSVT: statistic={w_cv_csvt.statistic}, p-value={w_cv_csvt.pvalue:.4f}")
        pairwise_results['CV_CSVT'] = {'test': 'Wilcoxon', 'statistic': w_cv_csvt.statistic, 'pvalue': w_cv_csvt.pvalue}

        # CV vs CSVS4
        w_cv_csvs4 = stats.wilcoxon(data['gmean_cv'], data['gmean_csvs4'])
        print(f"CV vs CSVS4: statistic={w_cv_csvs4.statistic}, p-value={w_cv_csvs4.pvalue:.4f}")
        pairwise_results['CV_CSVS4'] = {'test': 'Wilcoxon', 'statistic': w_cv_csvs4.statistic, 'pvalue': w_cv_csvs4.pvalue}

        # CSVT vs CSVS4
        w_csvt_csvs4 = stats.wilcoxon(data['gmean_csvt'], data['gmean_csvs4'])
        print(f"CSVT vs CSVS4: statistic={w_csvt_csvs4.statistic}, p-value={w_csvt_csvs4.pvalue:.4f}")
        pairwise_results['CSVT_CSVS4'] = {'test': 'Wilcoxon', 'statistic': w_csvt_csvs4.statistic, 'pvalue': w_csvt_csvs4.pvalue}

    else:
        print("Conclusion: Il n'existe pas de différence significative entre les trois groupes.")

# 5. Calculer des statistiques descriptives
print("\n--- Statistiques descriptives ---")
desc_stats = data[['gmean_cv', 'gmean_csvt', 'gmean_csvs4']].describe()
print(desc_stats)

# Calculer et afficher les différences moyennes entre les groupes
print("\n--- Différences moyennes entre les groupes ---")
diff_cv_csvt = np.mean(data['gmean_cv'] - data['gmean_csvt'])
diff_cv_csvs4 = np.mean(data['gmean_cv'] - data['gmean_csvs4'])
diff_csvt_csvs4 = np.mean(data['gmean_csvt'] - data['gmean_csvs4'])

print(f"Différence moyenne CV - CSVT: {diff_cv_csvt:.2f}")
print(f"Différence moyenne CV - CSVS4: {diff_cv_csvs4:.2f}")
print(f"Différence moyenne CSVT - CSVS4: {diff_csvt_csvs4:.2f}")

# 6. Visualiser les différences par système
plt.figure(figsize=(14, 8))
data_long = pd.melt(data, id_vars='systemes',
                   value_vars=['gmean_cv', 'gmean_csvt', 'gmean_csvs4'],
                   var_name='method', value_name='gmean')

sns.pointplot(x='systemes', y='gmean', hue='method', data=data_long, dodge=True)
plt.title("Comparaison des valeurs Gmean par système et méthode")
plt.xlabel("Systèmes")
plt.ylabel("Valeur Gmean (%)")
plt.xticks(rotation=45)
plt.legend(title='Méthode')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# 7. Afficher les différences pour chaque système
print("\n--- Valeurs et différences par système ---")
data['diff_cv_csvt'] = data['gmean_cv'] - data['gmean_csvt']
data['diff_cv_csvs4'] = data['gmean_cv'] - data['gmean_csvs4']
data['diff_csvt_csvs4'] = data['gmean_csvt'] - data['gmean_csvs4']

display(data[['systemes', 'gmean_cv', 'gmean_csvt', 'gmean_csvs4',
            'diff_cv_csvt', 'diff_cv_csvs4', 'diff_csvt_csvs4']])

# 8. Impact des corrections pour tests multiples
if len(pairwise_results) > 0:  # Vérifiez si des tests ont été effectués
    print("\n--- Correction de Bonferroni pour tests multiples ---")
    corrected_alpha = alpha / 3  # Trois comparaisons
    print(f"Seuil alpha corrigé: {corrected_alpha:.5f}")

    for comparison, result in pairwise_results.items():
        is_significant = result['pvalue'] < corrected_alpha
        print(f"{comparison}: significatif à {corrected_alpha:.5f}? {'Oui' if is_significant else 'Non'}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import seaborn as sns
from scipy.stats import shapiro, friedmanchisquare, f_oneway
from statsmodels.stats.anova import AnovaRM

# Création du dataframe avec les données du tableau
systemes = ["Camel", "Ivy", "Jedit", "Log4j", "Lucene", "Synapse", "Xerces", "Ant", "Poi", "Velocity", "Xalan"]
gmean_cv = [58.24, 70.89, 60.43, 55.25, 68.45, 63.15, 51.6, 70.23, 66.46, 71.33, 65.22]
gmean_csvt = [60.44, 72.72, 57.05, 53.61, 56.21, 69, 59.10, 70.60, 72.95, 59.47, 62.61]
gmean_csvs4 = [59.44, 73.8, 61.08, 54.28, 63.84, 71.04, 56.19, 68.61, 71.98, 56.52, 63.16]

# Création du dataframe
data = pd.DataFrame({
    'systemes': systemes,
    'gmean_cv': gmean_cv,
    'gmean_csvt': gmean_csvt,
    'gmean_csvs4': gmean_csvs4
})

# Affichage du dataframe
print("Données d'entrée:")
display(data)

# 1. Visualisation des données
plt.figure(figsize=(14, 6))

# Boxplot pour comparer les distributions
plt.subplot(1, 2, 1)
data_melted = pd.melt(data, id_vars='systemes',
                     value_vars=['gmean_cv', 'gmean_csvt', 'gmean_csvs4'],
                     var_name='method', value_name='gmean')
sns.boxplot(x='method', y='gmean', data=data_melted)
plt.title("Distribution des valeurs Gmean par méthode")
plt.xlabel("Méthode")
plt.ylabel("Valeur Gmean (%)")

# 2. Test de normalité avec Shapiro-Wilk
print("\n--- Test de normalité (Shapiro-Wilk) ---")
shapiro_cv = shapiro(data['gmean_cv'])
shapiro_csvt = shapiro(data['gmean_csvt'])
shapiro_csvs4 = shapiro(data['gmean_csvs4'])

print(f"Test de Shapiro-Wilk pour %Gmean cv: W={shapiro_cv.statistic:.4f}, p-value={shapiro_cv.pvalue:.4f}")
print(f"Test de Shapiro-Wilk pour %Gmean csvt: W={shapiro_csvt.statistic:.4f}, p-value={shapiro_csvt.pvalue:.4f}")
print(f"Test de Shapiro-Wilk pour %Gmean csvs4: W={shapiro_csvs4.statistic:.4f}, p-value={shapiro_csvs4.pvalue:.4f}")

# 3. QQ-plots pour vérifier visuellement la normalité
plt.subplot(1, 2, 2)
stats.probplot(data['gmean_cv'], dist="norm", plot=plt)
plt.title("QQ-Plot pour %Gmean cv")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
stats.probplot(data['gmean_csvt'], dist="norm", plot=plt)
plt.title("QQ-Plot pour %Gmean csvt")

plt.subplot(1, 2, 2)
stats.probplot(data['gmean_csvs4'], dist="norm", plot=plt)
plt.title("QQ-Plot pour %Gmean csvs4")
plt.tight_layout()
plt.show()

# 4. En fonction des résultats de normalité, choisir le test approprié
alpha = 0.05  # Seuil de significativité
all_normal = (shapiro_cv.pvalue > alpha and
              shapiro_csvt.pvalue > alpha and
              shapiro_csvs4.pvalue > alpha)

# Préparation des données pour les tests
df_long = pd.melt(data, id_vars='systemes',
                 value_vars=['gmean_cv', 'gmean_csvt', 'gmean_csvs4'],
                 var_name='method', value_name='gmean')

# Initialiser les variables pour stocker les résultats des tests
pairwise_results = {}

if all_normal:
    print("\nLes trois distributions semblent suivre une loi normale. Application de l'ANOVA pour mesures répétées:")

    # ANOVA pour mesures répétées
    aov = AnovaRM(data=df_long, depvar='gmean', subject='systemes', within=['method']).fit()
    print(aov.summary())

    if aov.anova_table['Pr > F'].iloc[0] < alpha:
        print("Conclusion: Il existe au moins une différence significative entre les trois groupes.")

        # Post-hoc tests: Pairwise t-tests
        print("\n--- Tests post-hoc: Comparaisons pairées (t-test apparié) ---")

        # CV vs CSVT
        t_cv_csvt = stats.ttest_rel(data['gmean_cv'], data['gmean_csvt'])
        print(f"CV vs CSVT: t={t_cv_csvt.statistic:.4f}, p-value={t_cv_csvt.pvalue:.4f}")
        pairwise_results['CV_CSVT'] = {'test': 't-test', 'statistic': t_cv_csvt.statistic, 'pvalue': t_cv_csvt.pvalue}

        # CV vs CSVS4
        t_cv_csvs4 = stats.ttest_rel(data['gmean_cv'], data['gmean_csvs4'])
        print(f"CV vs CSVS4: t={t_cv_csvs4.statistic:.4f}, p-value={t_cv_csvs4.pvalue:.4f}")
        pairwise_results['CV_CSVS4'] = {'test': 't-test', 'statistic': t_cv_csvs4.statistic, 'pvalue': t_cv_csvs4.pvalue}

        # CSVT vs CSVS4
        t_csvt_csvs4 = stats.ttest_rel(data['gmean_csvt'], data['gmean_csvs4'])
        print(f"CSVT vs CSVS4: t={t_csvt_csvs4.statistic:.4f}, p-value={t_csvt_csvs4.pvalue:.4f}")
        pairwise_results['CSVT_CSVS4'] = {'test': 't-test', 'statistic': t_csvt_csvs4.statistic, 'pvalue': t_csvt_csvs4.pvalue}

    else:
        print("Conclusion: Il n'existe pas de différence significative entre les trois groupes.")

else:
    print("\nAu moins une des distributions ne suit pas une loi normale. Application du test de Friedman:")

    # Test de Friedman (test non paramétrique pour k échantillons appariés)
    friedman_test = friedmanchisquare(data['gmean_cv'], data['gmean_csvt'], data['gmean_csvs4'])
    print(f"Test de Friedman: chi2={friedman_test.statistic:.4f}, p-value={friedman_test.pvalue:.4f}")

    if friedman_test.pvalue < alpha:
        print("Conclusion: Il existe au moins une différence significative entre les trois groupes.")

        # Post-hoc tests: Wilcoxon signed-rank test
        print("\n--- Tests post-hoc: Comparaisons pairées (test de Wilcoxon) ---")

        # CV vs CSVT
        w_cv_csvt = stats.wilcoxon(data['gmean_cv'], data['gmean_csvt'])
        print(f"CV vs CSVT: statistic={w_cv_csvt.statistic}, p-value={w_cv_csvt.pvalue:.4f}")
        pairwise_results['CV_CSVT'] = {'test': 'Wilcoxon', 'statistic': w_cv_csvt.statistic, 'pvalue': w_cv_csvt.pvalue}

        # CV vs CSVS4
        w_cv_csvs4 = stats.wilcoxon(data['gmean_cv'], data['gmean_csvs4'])
        print(f"CV vs CSVS4: statistic={w_cv_csvs4.statistic}, p-value={w_cv_csvs4.pvalue:.4f}")
        pairwise_results['CV_CSVS4'] = {'test': 'Wilcoxon', 'statistic': w_cv_csvs4.statistic, 'pvalue': w_cv_csvs4.pvalue}

        # CSVT vs CSVS4
        w_csvt_csvs4 = stats.wilcoxon(data['gmean_csvt'], data['gmean_csvs4'])
        print(f"CSVT vs CSVS4: statistic={w_csvt_csvs4.statistic}, p-value={w_csvt_csvs4.pvalue:.4f}")
        pairwise_results['CSVT_CSVS4'] = {'test': 'Wilcoxon', 'statistic': w_csvt_csvs4.statistic, 'pvalue': w_csvt_csvs4.pvalue}

    else:
        print("Conclusion: Il n'existe pas de différence significative entre les trois groupes.")

# 5. Calculer des statistiques descriptives
print("\n--- Statistiques descriptives ---")
desc_stats = data[['gmean_cv', 'gmean_csvt', 'gmean_csvs4']].describe()
print(desc_stats)

# Calculer et afficher les différences moyennes entre les groupes
print("\n--- Différences moyennes entre les groupes ---")
diff_cv_csvt = np.mean(data['gmean_cv'] - data['gmean_csvt'])
diff_cv_csvs4 = np.mean(data['gmean_cv'] - data['gmean_csvs4'])
diff_csvt_csvs4 = np.mean(data['gmean_csvt'] - data['gmean_csvs4'])

print(f"Différence moyenne CV - CSVT: {diff_cv_csvt:.2f}")
print(f"Différence moyenne CV - CSVS4: {diff_cv_csvs4:.2f}")
print(f"Différence moyenne CSVT - CSVS4: {diff_csvt_csvs4:.2f}")

# 6. Visualiser les différences par système
plt.figure(figsize=(14, 8))
data_long = pd.melt(data, id_vars='systemes',
                   value_vars=['gmean_cv', 'gmean_csvt', 'gmean_csvs4'],
                   var_name='method', value_name='gmean')

sns.pointplot(x='systemes', y='gmean', hue='method', data=data_long, dodge=True)
plt.title("Comparaison des valeurs Gmean par système et méthode")
plt.xlabel("Systèmes")
plt.ylabel("Valeur Gmean (%)")
plt.xticks(rotation=45)
plt.legend(title='Méthode')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# 7. Afficher les différences pour chaque système
print("\n--- Valeurs et différences par système ---")
data['diff_cv_csvt'] = data['gmean_cv'] - data['gmean_csvt']
data['diff_cv_csvs4'] = data['gmean_cv'] - data['gmean_csvs4']
data['diff_csvt_csvs4'] = data['gmean_csvt'] - data['gmean_csvs4']

display(data[['systemes', 'gmean_cv', 'gmean_csvt', 'gmean_csvs4',
            'diff_cv_csvt', 'diff_cv_csvs4', 'diff_csvt_csvs4']])

# 8. Impact des corrections pour tests multiples
if len(pairwise_results) > 0:  # Vérifiez si des tests ont été effectués
    print("\n--- Correction de Bonferroni pour tests multiples ---")
    corrected_alpha = alpha / 3  # Trois comparaisons
    print(f"Seuil alpha corrigé: {corrected_alpha:.5f}")

    for comparison, result in pairwise_results.items():
        is_significant = result['pvalue'] < corrected_alpha
        print(f"{comparison}: significatif à {corrected_alpha:.5f}? {'Oui' if is_significant else 'Non'}")

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# Création du DataFrame
data = {
    'Systemes': ['Ant', 'Camel', 'Ivy', 'Jedit', 'Log4j', 'Lucene', 'Poi', 'Synapse', 'Velocity', 'Xalan', 'Xerces'],
    'Gmean_cv_si': [68.71, 65.5, 72.57, 60.8, 54.43, 64.44, 65.49, 60.38, 63.76, 62.96, 45.53],
    'Gmean_cv_se': [70.23, 58.24, 70.89, 60.43, 55.25, 68.45, 66.46, 63.15, 71.33, 65.22, 51.6]
}

df = pd.DataFrame(data)

# Test de normalité (Shapiro-Wilk)
_, p_value_si = stats.shapiro(df['Gmean_cv_si'])
_, p_value_se = stats.shapiro(df['Gmean_cv_se'])

print("Test de normalité de Shapiro-Wilk:")
print(f"Gmean_cv_si: p-value = {p_value_si:.4f}")
print(f"Gmean_cv_se: p-value = {p_value_se:.4f}")

# Création des visualisations
plt.figure(figsize=(12, 4))

# QQ-plot pour Gmean_cv_si
plt.subplot(121)
stats.probplot(df['Gmean_cv_si'], dist="norm", plot=plt)
plt.title('Q-Q Plot Gmean_cv_si')

# QQ-plot pour Gmean_cv_se
plt.subplot(122)
stats.probplot(df['Gmean_cv_se'], dist="norm", plot=plt)
plt.title('Q-Q Plot Gmean_cv_se')
plt.tight_layout()
plt.show()

# Boxplots
plt.figure(figsize=(8, 6))
boxplot_data = [df['Gmean_cv_si'], df['Gmean_cv_se']]
plt.boxplot(boxplot_data, labels=['Gmean_cv_si', 'Gmean_cv_se'])
plt.title('Boxplot des distributions')
plt.ylabel('Valeurs')
plt.show()

# Test statistique
if p_value_si > 0.05 and p_value_se > 0.05:
    # Si les deux distributions sont normales, utiliser le t-test
    t_stat, p_value = stats.ttest_rel(df['Gmean_cv_si'], df['Gmean_cv_se'])
    test_name = "T-test apparié"
else:
    # Si au moins une distribution n'est pas normale, utiliser le test de Wilcoxon
    t_stat, p_value = stats.wilcoxon(df['Gmean_cv_si'], df['Gmean_cv_se'])
    test_name = "Test de Wilcoxon"

print(f"\nRésultats du {test_name}:")
print(f"Statistique = {t_stat:.4f}")
print(f"p-value = {p_value:.4f}")

# Statistiques descriptives
print("\nStatistiques descriptives:")
print(df[['Gmean_cv_si', 'Gmean_cv_se']].describe())

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# Création du DataFrame
data = {
    'Systemes': ['Ant', 'Camel', 'Ivy', 'Jedit', 'Log4j', 'Lucene', 'Poi', 'Synapse', 'Velocity', 'Xalan', 'Xerces'],
    'Gmean_csv': [69.16, 59.00, 74.68, 56.85, 54.92, 53.05, 68.27, 67.83, 54.40, 61.08, 60.17],
    'Gmean_csv_se': [70.60, 60.44, 72.72, 57.05, 53.61, 56.21, 72.95, 69.00, 59.47, 62.61, 59.10]
}

df = pd.DataFrame(data)

# Test de normalité (Shapiro-Wilk)
_, p_value_csv = stats.shapiro(df['Gmean_csv'])
_, p_value_se = stats.shapiro(df['Gmean_csv_se'])

print("Test de normalité de Shapiro-Wilk:")
print(f"Gmean_csv: p-value = {p_value_csv:.4f}")
print(f"Gmean_csv_se: p-value = {p_value_se:.4f}")

# Création des visualisations
plt.figure(figsize=(12, 4))

# QQ-plot pour Gmean_csv
plt.subplot(121)
stats.probplot(df['Gmean_csv'], dist="norm", plot=plt)
plt.title('Q-Q Plot Gmean_csv')

# QQ-plot pour Gmean_csv_se
plt.subplot(122)
stats.probplot(df['Gmean_csv_se'], dist="norm", plot=plt)
plt.title('Q-Q Plot Gmean_csv_se')
plt.tight_layout()
plt.show()

# Boxplots
plt.figure(figsize=(8, 6))
boxplot_data = [df['Gmean_csv'], df['Gmean_csv_se']]
plt.boxplot(boxplot_data, labels=['Gmean_csv', 'Gmean_csv_se'])
plt.title('Boxplot des distributions')
plt.ylabel('Valeurs')
plt.show()

# Test statistique
if p_value_csv > 0.05 and p_value_se > 0.05:
    # Si les deux distributions sont normales, utiliser le t-test
    t_stat, p_value = stats.ttest_rel(df['Gmean_csv'], df['Gmean_csv_se'])
    test_name = "T-test apparié"
else:
    # Si au moins une distribution n'est pas normale, utiliser le test de Wilcoxon
    t_stat, p_value = stats.wilcoxon(df['Gmean_csv'], df['Gmean_csv_se'])
    test_name = "Test de Wilcoxon"

print(f"\nRésultats du {test_name}:")
print(f"Statistique = {t_stat:.4f}")
print(f"p-value = {p_value:.4f}")

# Statistiques descriptives
print("\nStatistiques descriptives:")
print(df[['Gmean_csv', 'Gmean_csv_se']].describe())

# Calcul de la différence moyenne
mean_diff = (df['Gmean_csv_se'] - df['Gmean_csv']).mean()
print(f"\nDifférence moyenne (csv_se - csv) = {mean_diff:.2f}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import seaborn as sns
from scipy.stats.mstats import friedman
from scipy.stats import shapiro, ttest_rel, wilcoxon

# Création du dataframe avec les données du tableau
systemes = ["Ant", "Camel", "Ivy", "Jedit", "Log4j", "Lucene", "Poi", "Synapse", "Velocity", "Xalan", "Xerces"]
gmean_csv = [69.16, 59.00, 74.68, 56.85, 54.92, 53.05, 68.27, 67.83, 54.40, 61.08, 60.17]
gmean_csv_se = [70.60, 60.44, 72.72, 57.05, 53.61, 56.21, 72.95, 69.00, 59.47, 62.61, 59.10]

# Création du dataframe
data = pd.DataFrame({
    'systemes': systemes,
    'gmean_csv': gmean_csv,
    'gmean_csv_se': gmean_csv_se
})

# Affichage du dataframe
print("Données d'entrée:")
display(data)

# 1. Visualisation des données
plt.figure(figsize=(12, 5))

# Boxplot pour comparer les distributions
plt.subplot(1, 2, 1)
data_melted = pd.melt(data, id_vars='systemes', value_vars=['gmean_csv', 'gmean_csv_se'],
                     var_name='method', value_name='gmean')
sns.boxplot(x='method', y='gmean', data=data_melted)
plt.title("Distribution des valeurs Gmean par méthode")
plt.xlabel("Méthode")
plt.ylabel("Valeur Gmean (%)")

# 2. Test de normalité avec Shapiro-Wilk
print("\n--- Test de normalité (Shapiro-Wilk) ---")
shapiro_csv = shapiro(data['gmean_csv'])
shapiro_csv_se = shapiro(data['gmean_csv_se'])

print(f"Test de Shapiro-Wilk pour %Gmean csv: W={shapiro_csv.statistic:.4f}, p-value={shapiro_csv.pvalue:.4f}")
print(f"Test de Shapiro-Wilk pour %Gmean csv se: W={shapiro_csv_se.statistic:.4f}, p-value={shapiro_csv_se.pvalue:.4f}")

# 3. QQ-plots pour vérifier visuellement la normalité
plt.subplot(1, 2, 2)
stats.probplot(data['gmean_csv'], dist="norm", plot=plt)
plt.title("QQ-Plot pour %Gmean csv")

plt.figure(figsize=(6, 5))
stats.probplot(data['gmean_csv_se'], dist="norm", plot=plt)
plt.title("QQ-Plot pour %Gmean csv se")

plt.tight_layout()
plt.show()

# 4. Test statistique approprié
alpha = 0.05  # Seuil de significativité

if shapiro_csv.pvalue > alpha and shapiro_csv_se.pvalue > alpha:
    print("\nLes deux distributions semblent suivre une loi normale. Application du test t apparié:")
    t_test_result = ttest_rel(data['gmean_csv'], data['gmean_csv_se'])
    print(f"Test t apparié: t={t_test_result.statistic:.4f}, p-value={t_test_result.pvalue:.4f}")
    if t_test_result.pvalue < alpha:
        print("Conclusion: Il existe une différence significative entre les deux groupes.")
    else:
        print("Conclusion: Il n'existe pas de différence significative entre les deux groupes.")
else:
    print("\nAu moins une des distributions ne suit pas une loi normale. Application du test de Wilcoxon:")
    wilcox_test_result = wilcoxon(data['gmean_csv'], data['gmean_csv_se'])
    print(f"Test de Wilcoxon pour échantillons appariés: statistic={wilcox_test_result.statistic}, p-value={wilcox_test_result.pvalue:.4f}")
    if wilcox_test_result.pvalue < alpha:
        print("Conclusion: Il existe une différence significative entre les deux groupes.")
    else:
        print("Conclusion: Il n'existe pas de différence significative entre les deux groupes.")

# 5. Statistiques descriptives
print("\n--- Statistiques descriptives ---")
print(f"Moyenne %Gmean csv: {np.mean(data['gmean_csv']):.2f}")
print(f"Écart-type %Gmean csv: {np.std(data['gmean_csv'], ddof=1):.2f}")
print(f"Moyenne %Gmean csv se: {np.mean(data['gmean_csv_se']):.2f}")
print(f"Écart-type %Gmean csv se: {np.std(data['gmean_csv_se'], ddof=1):.2f}")
print(f"Différence moyenne (csv_se - csv): {np.mean(data['gmean_csv_se'] - data['gmean_csv']):.2f}")

# 6. Visualisation des différences
diff = data['gmean_csv_se'] - data['gmean_csv']
plt.figure(figsize=(12, 6))
plt.bar(data['systemes'], diff, color=np.where(diff >= 0, 'steelblue', 'tomato'))
plt.axhline(y=0, linestyle='--', color='black')
plt.title("Différence entre %Gmean csv se et %Gmean csv par système")
plt.xlabel("Systèmes")
plt.ylabel("Différence (csv_se - csv)")
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# 7. Différences système par système
print("\n--- Différences par système ---")
data['diff'] = data['gmean_csv_se'] - data['gmean_csv']
display(data[['systemes', 'gmean_csv', 'gmean_csv_se', 'diff']].sort_values(by='diff'))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import shapiro, f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy.stats import chi2
from sklearn.preprocessing import StandardScaler
from scipy.stats import friedmanchisquare

# Création du dataframe
systemes = ["Ant", "Camel", "Ivy", "Jedit", "Log4j", "Lucene", "Poi", "Synapse", "Velocity", "Xalan", "Xerces"]
gmean_cv_si = [68.71, 65.5, 72.57, 60.8, 54.43, 64.44, 65.49, 60.38, 63.76, 62.96, 45.53]
gmean_cv_se = [70.23, 58.24, 70.89, 60.43, 55.25, 68.45, 66.46, 63.15, 71.33, 65.22, 51.6]
gmean_csv = [69.16, 59.00, 74.68, 56.85, 54.92, 53.05, 68.27, 67.83, 54.4, 61.08, 60.17]
gmean_csv_se = [70.6, 60.44, 72.72, 57.05, 53.61, 56.21, 72.95, 69.0, 59.47, 62.61, 59.1]

data = pd.DataFrame({
    'systemes': systemes,
    'cv_si': gmean_cv_si,
    'cv_se': gmean_cv_se,
    'csv': gmean_csv,
    'csv_se': gmean_csv_se
})

# 1. Statistiques descriptives
print("Statistiques descriptives:")
stats_desc = data[['cv_si', 'cv_se', 'csv', 'csv_se']].describe()
print(stats_desc)

# 2. Tests de normalité
print("\nTests de normalité (Shapiro-Wilk):")
for col in ['cv_si', 'cv_se', 'csv', 'csv_se']:
    stat, p_value = shapiro(data[col])
    print(f"{col}: W={stat:.4f}, p-value={p_value:.4f}")

# 3. Visualisations
plt.figure(figsize=(12, 6))
data_melted = pd.melt(data, id_vars='systemes', value_vars=['cv_si', 'cv_se', 'csv', 'csv_se'],
                     var_name='method', value_name='gmean')
sns.boxplot(x='method', y='gmean', data=data_melted)
plt.title("Distribution des valeurs Gmean par méthode")
plt.xlabel("Méthode")
plt.ylabel("Valeur Gmean (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 4. Test de Friedman
print("\nTest de Friedman:")
statistic, p_value = friedmanchisquare(data['cv_si'], data['cv_se'], data['csv'], data['csv_se'])
print(f"Statistique={statistic:.4f}, p-value={p_value:.4f}")

# 5. ANOVA à un facteur
print("\nANOVA à un facteur:")
f_stat, p_value = f_oneway(data['cv_si'], data['cv_se'], data['csv'], data['csv_se'])
print(f"F={f_stat:.4f}, p-value={p_value:.4f}")

# 6. Test post-hoc de Tukey
print("\nTest post-hoc de Tukey:")
tukey = pairwise_tukeyhsd(data_melted['gmean'], data_melted['method'])
print(tukey)

# 7. Visualisation des différences par système
plt.figure(figsize=(15, 6))
data_long = pd.melt(data, id_vars='systemes', value_vars=['cv_si', 'cv_se', 'csv', 'csv_se'])
sns.lineplot(data=data_long, x='systemes', y='value', hue='variable', marker='o')
plt.title("Comparaison des méthodes par système")
plt.xlabel("Systèmes")
plt.ylabel("Valeur Gmean (%)")
plt.xticks(rotation=45)
plt.legend(title="Méthode")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 8. Matrice de corrélation
correlation_matrix = data[['cv_si', 'cv_se', 'csv', 'csv_se']].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, center=0)
plt.title("Matrice de corrélation entre les méthodes")
plt.tight_layout()
plt.show()

# 9. Calcul des différences moyennes entre les méthodes
print("\nDifférences moyennes entre les méthodes:")
methods = ['cv_si', 'cv_se', 'csv', 'csv_se']
for i in range(len(methods)):
    for j in range(i+1, len(methods)):
        diff = data[methods[j]].mean() - data[methods[i]].mean()
        print(f"{methods[j]} - {methods[i]}: {diff:.2f}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import shapiro, f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy.stats import chi2
from sklearn.preprocessing import StandardScaler
from scipy.stats import friedmanchisquare

# Création du dataframe
systemes = ["Ant", "Camel", "Ivy", "Jedit", "Log4j", "Lucene", "Poi", "Synapse", "Velocity", "Xalan", "Xerces"]
gmean_cv_si = [68.71, 65.5, 72.57, 60.8, 54.43, 64.44, 65.49, 60.38, 63.76, 62.96, 45.53]
gmean_cv_se = [70.23, 58.24, 70.89, 60.43, 55.25, 68.45, 66.46, 63.15, 71.33, 65.22, 51.6]
gmean_csv = [69.16, 59.00, 74.68, 56.85, 54.92, 53.05, 68.27, 67.83, 54.4, 61.08, 60.17]
gmean_csv_se = [70.6, 60.44, 72.72, 57.05, 53.61, 56.21, 72.95, 69.0, 59.47, 62.61, 59.1]

data = pd.DataFrame({
    'systemes': systemes,
    'cv_si': gmean_cv_si,
    'cv_se': gmean_cv_se,
    'csv': gmean_csv,
    'csv_se': gmean_csv_se
})

# 1. Statistiques descriptives
print("Statistiques descriptives:")
stats_desc = data[['cv_si', 'cv_se', 'csv', 'csv_se']].describe()
print(stats_desc)

# 2. Tests de normalité
print("\nTests de normalité (Shapiro-Wilk):")
for col in ['cv_si', 'cv_se', 'csv', 'csv_se']:
    stat, p_value = shapiro(data[col])
    print(f"{col}: W={stat:.4f}, p-value={p_value:.4f}")

# 3. Visualisations
plt.figure(figsize=(12, 6))
data_melted = pd.melt(data, id_vars='systemes', value_vars=['cv_si', 'cv_se', 'csv', 'csv_se'],
                     var_name='method', value_name='gmean')
sns.boxplot(x='method', y='gmean', data=data_melted)
plt.title("Distribution des valeurs Gmean par méthode")
plt.xlabel("Méthode")
plt.ylabel("Valeur Gmean (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 4. Test de Friedman
print("\nTest de Friedman:")
statistic, p_value = friedmanchisquare(data['cv_si'], data['cv_se'], data['csv'], data['csv_se'])
print(f"Statistique={statistic:.4f}, p-value={p_value:.4f}")

# 5. ANOVA à un facteur
print("\nANOVA à un facteur:")
f_stat, p_value = f_oneway(data['cv_si'], data['cv_se'], data['csv'], data['csv_se'])
print(f"F={f_stat:.4f}, p-value={p_value:.4f}")

# 6. Test post-hoc de Tukey
print("\nTest post-hoc de Tukey:")
tukey = pairwise_tukeyhsd(data_melted['gmean'], data_melted['method'])
print(tukey)

# 7. Visualisation des différences par système
plt.figure(figsize=(15, 6))
data_long = pd.melt(data, id_vars='systemes', value_vars=['cv_si', 'cv_se', 'csv', 'csv_se'])
sns.lineplot(data=data_long, x='systemes', y='value', hue='variable', marker='o')
plt.title("Comparaison des méthodes par système")
plt.xlabel("Systèmes")
plt.ylabel("Valeur Gmean (%)")
plt.xticks(rotation=45)
plt.legend(title="Méthode")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 8. Matrice de corrélation
correlation_matrix = data[['cv_si', 'cv_se', 'csv', 'csv_se']].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, center=0)
plt.title("Matrice de corrélation entre les méthodes")
plt.tight_layout()
plt.show()

# 9. Calcul des différences moyennes entre les méthodes
print("\nDifférences moyennes entre les méthodes:")
methods = ['cv_si', 'cv_se', 'csv', 'csv_se']
for i in range(len(methods)):
    for j in range(i+1, len(methods)):
        diff = data[methods[j]].mean() - data[methods[i]].mean()
        print(f"{methods[j]} - {methods[i]}: {diff:.2f}")

In [ ]:
!pip install scikit-posthocs

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import scikit_posthocs as sp
from statsmodels.stats.anova import AnovaRM

# Créer le DataFrame à partir des données fournies
systems = ['Camel', 'Ivy', 'Jedit', 'Log4j', 'Lucene', 'Synapse', 'Xerces', 'Ant', 'Poi', 'Velocity', 'Xalan']
smote = [59, 74.68, 56.85, 54.92, 53.05, 67.83, 60.17, 69.16, 68.27, 54.4, 61.08]
random_under = [60.44, 72.72, 57.05, 53.61, 56.21, 69, 59.1, 70.6, 72.95, 59.47, 62.61]
outlier_removal = [58.68, 73.25, 58.52, 51.75, 59.6, 69.85, 53.63, 71.06, 72.19, 60.56, 62.61]

# Créer le DataFrame
df = pd.DataFrame({
    'Système': systems,
    'SMOTE': smote,
    'Sous-échantillonnage': random_under,
    'Suppression outliers': outlier_removal
})

print("Données utilisées pour l'analyse:")
print(df)

# Calculer les statistiques descriptives
print("\nStatistiques descriptives:")
descriptive_stats = df.describe()
print(descriptive_stats)

# Calculer les moyennes (qui devraient correspondre à celles dans le document)
print("\nMoyennes par méthode:")
means = df[['SMOTE', 'Sous-échantillonnage', 'Suppression outliers']].mean()
print(means)

# Réorganiser les données pour l'ANOVA à mesures répétées
df_long = pd.melt(df, id_vars=['Système'],
                 value_vars=['SMOTE', 'Sous-échantillonnage', 'Suppression outliers'],
                 var_name='Méthode', value_name='Gmean')

# 1. Vérifier la normalité des différences (pour tester si ANOVA est appropriée)
print("\n1. Test de normalité (Shapiro-Wilk) pour chaque méthode:")
for method in ['SMOTE', 'Sous-échantillonnage', 'Suppression outliers']:
    stat, p = stats.shapiro(df[method])
    print(f"{method}: W={stat:.4f}, p-value={p:.4f}")
    if p > 0.05:
        print(f"  Les données semblent suivre une distribution normale (p > 0.05)")
    else:
        print(f"  Les données ne semblent pas suivre une distribution normale (p <= 0.05)")

# 2. ANOVA à mesures répétées
print("\n2. ANOVA à mesures répétées:")
try:
    # Tenter d'exécuter ANOVA à mesures répétées
    aov = AnovaRM(df_long, 'Gmean', 'Système', within=['Méthode']).fit()
    print(aov.anova_table)

    # Interpréter les résultats
    p_value = aov.anova_table['Pr > F']['Méthode']
    if p_value < 0.05:
        print(f"Différence significative détectée entre les méthodes (p={p_value:.4f} < 0.05)")
    else:
        print(f"Pas de différence significative entre les méthodes (p={p_value:.4f} >= 0.05)")

except Exception as e:
    print(f"Erreur lors de l'ANOVA à mesures répétées: {e}")
    print("Utilisation du test de Friedman comme alternative non paramétrique.")

# 3. Test de Friedman (alternative non paramétrique à l'ANOVA à mesures répétées)
print("\n3. Test de Friedman (test non paramétrique pour données appariées):")
friedman_stat, friedman_p = stats.friedmanchisquare(df['SMOTE'], df['Sous-échantillonnage'], df['Suppression outliers'])
print(f"Chi2={friedman_stat:.4f}, p-value={friedman_p:.4f}")

if friedman_p < 0.05:
    print(f"Différence significative détectée entre les méthodes (p={friedman_p:.4f} < 0.05)")

    # Test post-hoc de Nemenyi
    print("\n4. Tests post-hoc (Nemenyi/Conover):")
    posthoc = sp.posthoc_nemenyi_friedman(df[['SMOTE', 'Sous-échantillonnage', 'Suppression outliers']])
    print("Matrice de p-values pour les comparaisons par paires:")
    print(posthoc)

    # Test de Wilcoxon par paires avec correction de Bonferroni
    print("\n5. Tests de Wilcoxon par paires (avec correction de Bonferroni):")
    # Correction de Bonferroni: alpha / nombre de tests
    alpha_corrigé = 0.05 / 3  # 3 paires à comparer

    # SMOTE vs Sous-échantillonnage
    stat, p = stats.wilcoxon(df['SMOTE'], df['Sous-échantillonnage'])
    print(f"SMOTE vs Sous-échantillonnage: p-value={p:.4f}", end=' ')
    print(f"{'(significatif)' if p < alpha_corrigé else '(non significatif)'}")

    # SMOTE vs Suppression outliers
    stat, p = stats.wilcoxon(df['SMOTE'], df['Suppression outliers'])
    print(f"SMOTE vs Suppression outliers: p-value={p:.4f}", end=' ')
    print(f"{'(significatif)' if p < alpha_corrigé else '(non significatif)'}")

    # Sous-échantillonnage vs Suppression outliers
    stat, p = stats.wilcoxon(df['Sous-échantillonnage'], df['Suppression outliers'])
    print(f"Sous-échantillonnage vs Suppression outliers: p-value={p:.4f}", end=' ')
    print(f"{'(significatif)' if p < alpha_corrigé else '(non significatif)'}")
else:
    print(f"Pas de différence significative entre les méthodes (p={friedman_p:.4f} >= 0.05)")

# Visualisations
# 1. Boxplot pour comparer les distributions
plt.figure(figsize=(12, 6))
ax = sns.boxplot(x='Méthode', y='Gmean', data=df_long)
plt.title('Distribution des scores Gmean par méthode')
plt.grid(True, linestyle='--', alpha=0.7)

# Ajouter les points individuels
sns.stripplot(x='Méthode', y='Gmean', data=df_long,
              color='black', alpha=0.5, jitter=True)

plt.tight_layout()

# 2. Graphique des moyennes avec intervalles de confiance
plt.figure(figsize=(12, 6))
ax = sns.barplot(x='Méthode', y='Gmean', data=df_long, errorbar=('ci', 95), capsize=0.2)
plt.title('Scores Gmean moyens avec intervalles de confiance à 95%')
plt.ylim(55, 65)  # Ajuster l'échelle pour mieux voir les différences
plt.grid(True, linestyle='--', alpha=0.7, axis='y')

# Ajouter les valeurs sur les barres
for i, p in enumerate(ax.patches):
    height = p.get_height()
    ax.text(p.get_x() + p.get_width()/2., height + 0.1,
            f'{height:.2f}%',
            ha='center', fontweight='bold')

plt.tight_layout()

# 3. Graphique de profil pour chaque système
plt.figure(figsize=(14, 8))
for i, system in enumerate(systems):
    system_data = df[df['Système'] == system]
    values = system_data[['SMOTE', 'Sous-échantillonnage', 'Suppression outliers']].values[0]
    plt.plot(['SMOTE', 'Sous-échantillonnage', 'Suppression outliers'],
             values, 'o-', label=system)

plt.title('Profil de performance pour chaque système à travers les méthodes')
plt.ylabel('Score Gmean (%)')
plt.grid(True, linestyle='--', alpha=0.4)
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))

plt.tight_layout()
plt.show()

print("\n=== Conclusion ===")
print("L'analyse a été effectuée en utilisant des tests statistiques appropriés pour données appariées.")
print("Les tests de Friedman et de Wilcoxon par paires sont particulièrement appropriés car ils ne")
print("nécessitent pas l'hypothèse de normalité des données.")
print("Si l'écart de ~1.3% entre les moyennes est statistiquement significatif dépend des résultats des tests.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import scikit_posthocs as sp
from statsmodels.stats.anova import AnovaRM

# Créer le DataFrame à partir des données fournies
systems = ['Camel', 'Ivy', 'Jedit', 'Log4j', 'Lucene', 'Synapse', 'Xerces']

# Données pour chaque configuration
# 1. Tous les systèmes avec suppression des outliers
all_systems_outlier_removal = [58.68, 73.25, 58.52, 51.75, 59.6, 69.85, 53.63]

# 2. Sans les 4 systèmes avec suppression des outliers
without_4_outlier_removal = [59.55, 69.29, 57.64, 52.7, 60.26, 70.64, 57.64]

# 3. Sans les 4 systèmes sous-échantillonnage aléatoire
without_4_random_undersampling = [59.44, 73.8, 61.08, 54.28, 63.84, 71.04, 56.19]

# 4. Tous les systèmes avec sous-échantillonnage aléatoire
all_systems_random_undersampling = [60.44, 72.72, 57.05, 53.61, 56.21, 69, 59.1]

# 5. Sans les 4 systèmes avec SMOTE
without_4_smote = [60.48, 69.45, 57.54, 59.71, 58.43, 60.49, 55.93]

# 6. Tous les systèmes avec SMOTE
all_systems_smote = [59, 74.68, 56.85, 54.92, 53.05, 67.83, 60.17]

# Créer le DataFrame
df = pd.DataFrame({
    'Système': systems,
    'Tous systèmes + outlier removal': all_systems_outlier_removal,
    'Sans 4 systèmes + outlier removal': without_4_outlier_removal,
    'Sans 4 systèmes + sous-échantillonnage': without_4_random_undersampling,
    'Tous systèmes + sous-échantillonnage': all_systems_random_undersampling,
    'Sans 4 systèmes + SMOTE': without_4_smote,
    'Tous systèmes + SMOTE': all_systems_smote
})

# Ajouter la moyenne par système
df['Moyenne par système'] = df.iloc[:, 1:7].mean(axis=1)

print("Données utilisées pour l'analyse:")
print(df)

# Calculer les moyennes par configuration (colonne)
print("\nMoyennes par configuration:")
config_means = df.iloc[:, 1:7].mean()
print(config_means)

# Statistiques descriptives
print("\nStatistiques descriptives:")
descriptive_stats = df.iloc[:, 1:7].describe()
print(descriptive_stats)

# Réorganiser les données pour l'analyse statistique
df_long = pd.melt(df, id_vars=['Système'],
                 value_vars=df.columns[1:7],
                 var_name='Configuration', value_name='Gmean')

# 1. Vérifier la normalité des différences
print("\n1. Test de normalité (Shapiro-Wilk) pour chaque configuration:")
for config in df.columns[1:7]:
    stat, p = stats.shapiro(df[config])
    print(f"{config}: W={stat:.4f}, p-value={p:.4f}")
    if p > 0.05:
        print(f"  Les données semblent suivre une distribution normale (p > 0.05)")
    else:
        print(f"  Les données ne semblent pas suivre une distribution normale (p <= 0.05)")

# 2. ANOVA à mesures répétées (données appariées)
print("\n2. ANOVA à mesures répétées:")
try:
    # Utiliser une ANOVA à mesures répétées
    aov = AnovaRM(df_long, 'Gmean', 'Système', within=['Configuration']).fit()
    print(aov.anova_table)

    # Interpréter les résultats
    p_value = aov.anova_table['Pr > F']['Configuration']
    if p_value < 0.05:
        print(f"Différence significative détectée entre les configurations (p={p_value:.4f} < 0.05)")
    else:
        print(f"Pas de différence significative entre les configurations (p={p_value:.4f} >= 0.05)")

except Exception as e:
    print(f"Erreur lors de l'ANOVA à mesures répétées: {e}")
    print("Utilisation du test de Friedman comme alternative non paramétrique.")

# 3. Test de Friedman (alternative non paramétrique pour données appariées)
print("\n3. Test de Friedman (test non paramétrique pour données appariées):")
friedman_stat, friedman_p = stats.friedmanchisquare(*[df[col] for col in df.columns[1:7]])
print(f"Chi2={friedman_stat:.4f}, p-value={friedman_p:.4f}")

if friedman_p < 0.05:
    print(f"Différence significative détectée entre les configurations (p={friedman_p:.4f} < 0.05)")

    # Test post-hoc de Nemenyi pour identifier quelles paires de configurations diffèrent
    print("\n4. Test post-hoc (Nemenyi/Conover):")
    posthoc = sp.posthoc_nemenyi_friedman(df[df.columns[1:7]])
    print("Matrice de p-values pour les comparaisons par paires:")
    # Renommer les indices pour plus de clarté
    posthoc.index = df.columns[1:7]
    posthoc.columns = df.columns[1:7]
    print(posthoc)

    # Identifier les paires significativement différentes (p < 0.05)
    print("\nPaires de configurations significativement différentes (p < 0.05):")
    significant_pairs = []
    for i, row in enumerate(df.columns[1:7]):
        for j, col in enumerate(df.columns[1:7]):
            if i < j and posthoc.iloc[i, j] < 0.05:
                significant_pairs.append((row, col, posthoc.iloc[i, j]))

    if significant_pairs:
        for pair in significant_pairs:
            print(f"- {pair[0]} vs {pair[1]}: p={pair[2]:.4f}")
    else:
        print("Aucune paire n'est significativement différente au niveau alpha=0.05")
        # Vérifier avec un niveau alpha plus élevé
        print("\nPaires potentiellement différentes (p < 0.1):")
        trend_pairs = []
        for i, row in enumerate(df.columns[1:7]):
            for j, col in enumerate(df.columns[1:7]):
                if i < j and posthoc.iloc[i, j] < 0.1:
                    trend_pairs.append((row, col, posthoc.iloc[i, j]))

        if trend_pairs:
            for pair in trend_pairs:
                print(f"- {pair[0]} vs {pair[1]}: p={pair[2]:.4f}")
        else:
            print("Aucune paire n'est différente même au niveau alpha=0.1")
else:
    print(f"Pas de différence significative entre les configurations (p={friedman_p:.4f} >= 0.05)")

    # Examiner les différences sur la base des moyennes
    print("\nClassement des configurations basé sur les moyennes (pas de différence statistique significative):")
    sorted_means = config_means.sort_values(ascending=False)
    for i, (config, mean) in enumerate(sorted_means.items()):
        print(f"{i+1}. {config}: {mean:.2f}%")

# 4. Analyse des différences entre "Tous systèmes" vs "Sans 4 systèmes"
print("\n5. Analyse de l'effet de la sélection des systèmes:")
# Comparer les paires de configurations qui diffèrent uniquement par la sélection des systèmes
pairs_to_compare = [
    ('Tous systèmes + outlier removal', 'Sans 4 systèmes + outlier removal'),
    ('Tous systèmes + sous-échantillonnage', 'Sans 4 systèmes + sous-échantillonnage'),
    ('Tous systèmes + SMOTE', 'Sans 4 systèmes + SMOTE')
]

for pair in pairs_to_compare:
    stat, p = stats.wilcoxon(df[pair[0]], df[pair[1]])
    print(f"Comparaison {pair[0]} vs {pair[1]}:")
    print(f"  Test de Wilcoxon: p-value={p:.4f}")
    print(f"  Moyenne {pair[0]}: {df[pair[0]].mean():.2f}%")
    print(f"  Moyenne {pair[1]}: {df[pair[1]].mean():.2f}%")
    print(f"  Différence: {abs(df[pair[0]].mean() - df[pair[1]].mean()):.2f}%")
    if p < 0.05:
        print(f"  Différence significative (p < 0.05)")
    else:
        print(f"  Pas de différence significative (p >= 0.05)")

# 5. Analyse des différences entre méthodes de rééquilibrage
print("\n6. Analyse de l'effet des méthodes de rééquilibrage:")
# Comparer les paires de configurations qui diffèrent uniquement par la méthode de rééquilibrage
for sys_set in ['Tous systèmes', 'Sans 4 systèmes']:
    print(f"\nPour {sys_set}:")
    methods = [col for col in df.columns[1:7] if sys_set in col]

    for i, method1 in enumerate(methods):
        for j, method2 in enumerate(methods):
            if i < j:
                stat, p = stats.wilcoxon(df[method1], df[method2])
                print(f"Comparaison {method1} vs {method2}:")
                print(f"  Test de Wilcoxon: p-value={p:.4f}")
                print(f"  Moyenne {method1}: {df[method1].mean():.2f}%")
                print(f"  Moyenne {method2}: {df[method2].mean():.2f}%")
                print(f"  Différence: {abs(df[method1].mean() - df[method2].mean()):.2f}%")
                if p < 0.05:
                    print(f"  Différence significative (p < 0.05)")
                else:
                    print(f"  Pas de différence significative (p >= 0.05)")

# Visualisations
# 1. Boxplot pour comparer les distributions
plt.figure(figsize=(14, 8))
ax = sns.boxplot(x='Configuration', y='Gmean', data=df_long)
plt.title('Distribution des scores Gmean par configuration')
plt.xticks(rotation=45, ha='right')
plt.grid(True, linestyle='--', alpha=0.7)

# Ajouter les points individuels
sns.stripplot(x='Configuration', y='Gmean', data=df_long,
              color='black', alpha=0.5, jitter=True)

plt.tight_layout()

# 2. Graphique des moyennes avec intervalles de confiance
plt.figure(figsize=(14, 8))
ax = sns.barplot(x='Configuration', y='Gmean', data=df_long, errorbar=('ci', 95), capsize=0.2)
plt.title('Scores Gmean moyens avec intervalles de confiance à 95%')
plt.xticks(rotation=45, ha='right')
plt.ylim(58, 64)  # Ajuster l'échelle pour mieux voir les différences
plt.grid(True, linestyle='--', alpha=0.7, axis='y')

# Ajouter les valeurs sur les barres
for i, p in enumerate(ax.patches):
    height = p.get_height()
    ax.text(p.get_x() + p.get_width()/2., height + 0.1,
            f'{height:.2f}%',
            ha='center', fontweight='bold')

plt.tight_layout()

# 3. Graphique de profil pour chaque système
plt.figure(figsize=(14, 8))
for i, system in enumerate(systems):
    system_data = df[df['Système'] == system]
    values = system_data[df.columns[1:7]].values[0]
    plt.plot(df.columns[1:7], values, 'o-', label=system)

plt.title('Profil de performance pour chaque système à travers les configurations')
plt.ylabel('Score Gmean (%)')
plt.xticks(rotation=45, ha='right')
plt.grid(True, linestyle='--', alpha=0.4)
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))

plt.tight_layout()
plt.show()

print("\n=== Conclusion ===")
print("L'analyse a été effectuée en utilisant des tests statistiques appropriés pour")
print("données appariées afin de déterminer s'il existe des différences significatives")
print("entre les différentes configurations testées.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import scikit_posthocs as sp
from statsmodels.stats.anova import AnovaRM

# Créer le DataFrame à partir des données fournies
systems = ['Ant', 'Poi', 'Velocity', 'Xalan']

# Données pour chaque configuration
# 1. Tous les systèmes avec suppression des outliers
all_systems_outlier_removal = [71.06, 72.19, 60.56, 62.61]

# 2. Juste les 4 systèmes avec suppression des outliers
just_4_outlier_removal = [67.65, 72.16, 59.59, 62.59]

# 3. Juste les 4 systèmes sous-échantillonnage aléatoire
just_4_random_undersampling = [68.61, 71.98, 56.52, 63.16]

# 4. Tous les systèmes avec sous-échantillonnage aléatoire
all_systems_random_undersampling = [70.60, 72.95, 59.47, 62.61]

# 5. Juste les 4 systèmes avec SMOTE
just_4_smote = [69.09, 66.12, 58.28, 64.71]

# 6. Tous les systèmes avec SMOTE
all_systems_smote = [69.16, 68.27, 54.40, 61.08]

# Créer le DataFrame
df = pd.DataFrame({
    'Système': systems,
    'Tous systèmes + outlier removal': all_systems_outlier_removal,
    'Juste 4 systèmes + outlier removal': just_4_outlier_removal,
    'Juste 4 systèmes + sous-échantillonnage': just_4_random_undersampling,
    'Tous systèmes + sous-échantillonnage': all_systems_random_undersampling,
    'Juste 4 systèmes + SMOTE': just_4_smote,
    'Tous systèmes + SMOTE': all_systems_smote
})

# Ajouter la moyenne par système
df['Moyenne par système'] = df.iloc[:, 1:4].mean(axis=1)

print("Données utilisées pour l'analyse:")
print(df)

# Calculer les moyennes par configuration (colonne)
print("\nMoyennes par configuration:")
config_means = df.iloc[:, 1:4].mean()
print(config_means)

# Statistiques descriptives
print("\nStatistiques descriptives:")
descriptive_stats = df.iloc[:, 1:4].describe()
print(descriptive_stats)

# Réorganiser les données pour l'analyse statistique
df_long = pd.melt(df, id_vars=['Système'],
                 value_vars=df.columns[1:4],
                 var_name='Configuration', value_name='Gmean')

# 1. Vérifier la normalité des différences
print("\n1. Test de normalité (Shapiro-Wilk) pour chaque configuration:")
for config in df.columns[1:4]:
    stat, p = stats.shapiro(df[config])
    print(f"{config}: W={stat:.4f}, p-value={p:.4f}")
    if p > 0.05:
        print(f"  Les données semblent suivre une distribution normale (p > 0.05)")
    else:
        print(f"  Les données ne semblent pas suivre une distribution normale (p <= 0.05)")

# 2. ANOVA à mesures répétées (données appariées)
print("\n2. ANOVA à mesures répétées:")
try:
    # Utiliser une ANOVA à mesures répétées
    aov = AnovaRM(df_long, 'Gmean', 'Système', within=['Configuration']).fit()
    print(aov.anova_table)

    # Interpréter les résultats
    p_value = aov.anova_table['Pr > F']['Configuration']
    if p_value < 0.05:
        print(f"Différence significative détectée entre les configurations (p={p_value:.4f} < 0.05)")
    else:
        print(f"Pas de différence significative entre les configurations (p={p_value:.4f} >= 0.05)")

except Exception as e:
    print(f"Erreur lors de l'ANOVA à mesures répétées: {e}")
    print("Utilisation du test de Friedman comme alternative non paramétrique.")

# 3. Test de Friedman (alternative non paramétrique pour données appariées)
print("\n3. Test de Friedman (test non paramétrique pour données appariées):")
friedman_stat, friedman_p = stats.friedmanchisquare(*[df[col] for col in df.columns[1:7]])
print(f"Chi2={friedman_stat:.4f}, p-value={friedman_p:.4f}")

if friedman_p < 0.05:
    print(f"Différence significative détectée entre les configurations (p={friedman_p:.4f} < 0.05)")

    # Test post-hoc de Nemenyi pour identifier quelles paires de configurations diffèrent
    print("\n4. Test post-hoc (Nemenyi/Conover):")
    posthoc = sp.posthoc_nemenyi_friedman(df[df.columns[1:7]])
    print("Matrice de p-values pour les comparaisons par paires:")
    # Renommer les indices pour plus de clarté
    posthoc.index = df.columns[1:4]
    posthoc.columns = df.columns[1:4]
    print(posthoc)

    # Identifier les paires significativement différentes (p < 0.05)
    print("\nPaires de configurations significativement différentes (p < 0.05):")
    significant_pairs = []
    for i, row in enumerate(df.columns[1:4]):
        for j, col in enumerate(df.columns[1:4]):
            if i < j and posthoc.iloc[i, j] < 0.05:
                significant_pairs.append((row, col, posthoc.iloc[i, j]))

    if significant_pairs:
        for pair in significant_pairs:
            print(f"- {pair[0]} vs {pair[1]}: p={pair[2]:.4f}")
    else:
        print("Aucune paire n'est significativement différente au niveau alpha=0.05")
        # Vérifier avec un niveau alpha plus élevé
        print("\nPaires potentiellement différentes (p < 0.1):")
        trend_pairs = []
        for i, row in enumerate(df.columns[1:4]):
            for j, col in enumerate(df.columns[1:4]):
                if i < j and posthoc.iloc[i, j] < 0.1:
                    trend_pairs.append((row, col, posthoc.iloc[i, j]))

        if trend_pairs:
            for pair in trend_pairs:
                print(f"- {pair[0]} vs {pair[1]}: p={pair[2]:.4f}")
        else:
            print("Aucune paire n'est différente même au niveau alpha=0.1")
else:
    print(f"Pas de différence significative entre les configurations (p={friedman_p:.4f} >= 0.05)")

    # Examiner les différences sur la base des moyennes
    print("\nClassement des configurations basé sur les moyennes (pas de différence statistique significative):")
    sorted_means = config_means.sort_values(ascending=False)
    for i, (config, mean) in enumerate(sorted_means.items()):
        print(f"{i+1}. {config}: {mean:.2f}%")

# 4. Analyse des différences entre "Tous systèmes" vs "Sans 4 systèmes"
print("\n5. Analyse de l'effet de la sélection des systèmes:")
# Comparer les paires de configurations qui diffèrent uniquement par la sélection des systèmes
pairs_to_compare = [
    ('Tous systèmes + outlier removal', 'Juste 4 systèmes + outlier removal'),
    ('Tous systèmes + sous-échantillonnage', 'Juste 4 systèmes + sous-échantillonnage'),
    ('Tous systèmes + SMOTE', 'Juste 4 systèmes + SMOTE')
]

for pair in pairs_to_compare:
    stat, p = stats.wilcoxon(df[pair[0]], df[pair[1]])
    print(f"Comparaison {pair[0]} vs {pair[1]}:")
    print(f"  Test de Wilcoxon: p-value={p:.4f}")
    print(f"  Moyenne {pair[0]}: {df[pair[0]].mean():.2f}%")
    print(f"  Moyenne {pair[1]}: {df[pair[1]].mean():.2f}%")
    print(f"  Différence: {abs(df[pair[0]].mean() - df[pair[1]].mean()):.2f}%")
    if p < 0.05:
        print(f"  Différence significative (p < 0.05)")
    else:
        print(f"  Pas de différence significative (p >= 0.05)")

# 5. Analyse des différences entre méthodes de rééquilibrage
print("\n6. Analyse de l'effet des méthodes de rééquilibrage:")
# Comparer les paires de configurations qui diffèrent uniquement par la méthode de rééquilibrage
for sys_set in ['Tous systèmes', 'Juste 4 systèmes']:
    print(f"\nPour {sys_set}:")
    methods = [col for col in df.columns[1:4] if sys_set in col]

    for i, method1 in enumerate(methods):
        for j, method2 in enumerate(methods):
            if i < j:
                stat, p = stats.wilcoxon(df[method1], df[method2])
                print(f"Comparaison {method1} vs {method2}:")
                print(f"  Test de Wilcoxon: p-value={p:.4f}")
                print(f"  Moyenne {method1}: {df[method1].mean():.2f}%")
                print(f"  Moyenne {method2}: {df[method2].mean():.2f}%")
                print(f"  Différence: {abs(df[method1].mean() - df[method2].mean()):.2f}%")
                if p < 0.05:
                    print(f"  Différence significative (p < 0.05)")
                else:
                    print(f"  Pas de différence significative (p >= 0.05)")

# Visualisations
# 1. Boxplot pour comparer les distributions
plt.figure(figsize=(14, 8))
ax = sns.boxplot(x='Configuration', y='Gmean', data=df_long)
plt.title('Distribution des scores Gmean par configuration')
plt.xticks(rotation=45, ha='right')
plt.grid(True, linestyle='--', alpha=0.7)

# Ajouter les points individuels
sns.stripplot(x='Configuration', y='Gmean', data=df_long,
              color='black', alpha=0.5, jitter=True)

plt.tight_layout()

# 2. Graphique des moyennes avec intervalles de confiance
plt.figure(figsize=(14, 8))
ax = sns.barplot(x='Configuration', y='Gmean', data=df_long, errorbar=('ci', 95), capsize=0.2)
plt.title('Scores Gmean moyens avec intervalles de confiance à 95%')
plt.xticks(rotation=45, ha='right')
plt.ylim(58, 64)  # Ajuster l'échelle pour mieux voir les différences
plt.grid(True, linestyle='--', alpha=0.7, axis='y')

# Ajouter les valeurs sur les barres
for i, p in enumerate(ax.patches):
    height = p.get_height()
    ax.text(p.get_x() + p.get_width()/2., height + 0.1,
            f'{height:.2f}%',
            ha='center', fontweight='bold')

plt.tight_layout()

# 3. Graphique de profil pour chaque système
plt.figure(figsize=(14, 8))
for i, system in enumerate(systems):
    system_data = df[df['Système'] == system]
    values = system_data[df.columns[1:4]].values[0]
    plt.plot(df.columns[1:4], values, 'o-', label=system)

plt.title('Profil de performance pour chaque système à travers les configurations')
plt.ylabel('Score Gmean (%)')
plt.xticks(rotation=45, ha='right')
plt.grid(True, linestyle='--', alpha=0.4)
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))

plt.tight_layout()
plt.show()

print("\n=== Conclusion ===")
print("L'analyse a été effectuée en utilisant des tests statistiques appropriés pour")
print("données appariées afin de déterminer s'il existe des différences significatives")
print("entre les différentes configurations testées.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import scikit_posthocs as sp
from statsmodels.stats.anova import AnovaRM

# Créer le DataFrame à partir des nouvelles données
systems = ['Ant', 'Camel', 'Ivy', 'Jedit', 'Log4j', 'Lucene',
           'Poi', 'Synapse', 'Velocity', 'Xalan', 'Xerces']

# Données pour chaque configuration (extraites du CV CSV.pdf)
intraversion_smote = [68.71, 65.5, 72.57, 60.8, 54.43, 64.44, 65.49, 60.38, 63.76, 62.96, 45.53]
intraversion_sous_echantillonnage = [70.23, 58.24, 70.89, 60.43, 55.25, 68.45, 66.46, 63.15, 71.33, 65.22, 51.6]
intraversion_outlier_removal = [63.67, 71.83, 60.97, 56.02, 65.79, 60.63, 47.14, 69.75, 67.35, 53.97, 60.87]
interversion_smote = [69.16, 59.0, 74.68, 56.85, 54.92, 53.05, 68.27, 67.83, 54.4, 61.08, 60.17]
interversion_sous_echantillonnage = [70.6, 60.44, 72.72, 57.05, 53.61, 56.21, 72.95, 69.0, 59.47, 62.61, 59.1]
interversion_outlier_removal = [58.68, 73.25, 58.52, 51.75, 59.6, 69.85, 53.63, 71.06, 72.19, 60.56, 62.61]

# Créer le DataFrame
df = pd.DataFrame({
    'Système': systems,
    'Intraversion SMOTE': intraversion_smote,
    'Intraversion sous-échantillonnage': intraversion_sous_echantillonnage,
    'Intraversion suppression outliers': intraversion_outlier_removal,
    'Interversion SMOTE': interversion_smote,
    'Interversion sous-échantillonnage': interversion_sous_echantillonnage,
    'Interversion suppression outliers': interversion_outlier_removal,
})

print("Données utilisées pour l'analyse:")
print(df)

# Calculer les moyennes par configuration (colonne)
print("\nMoyennes par configuration:")
config_means = df.iloc[:, 1:11].mean()
print(config_means)

# Statistiques descriptives
print("\nStatistiques descriptives:")
descriptive_stats = df.iloc[:, 1:11].describe()
print(descriptive_stats)

# Réorganiser les données pour l'analyse statistique
df_long = pd.melt(df, id_vars=['Système'],
                 value_vars=df.columns[1:11],
                 var_name='Configuration', value_name='Gmean')

# 1. Vérifier la normalité des différences
print("\n1. Test de normalité (Shapiro-Wilk) pour chaque configuration:")
for config in df.columns[1:11]:
    stat, p = stats.shapiro(df[config])
    print(f"{config}: W={stat:.4f}, p-value={p:.4f}")
    if p > 0.05:
        print(f"  Les données semblent suivre une distribution normale (p > 0.05)")
    else:
        print(f"  Les données ne semblent pas suivre une distribution normale (p <= 0.05)")

# 2. ANOVA à mesures répétées (données appariées)
print("\n2. ANOVA à mesures répétées:")
try:
    # Utiliser une ANOVA à mesures répétées
    aov = AnovaRM(df_long, 'Gmean', 'Système', within=['Configuration']).fit()
    print(aov.anova_table)

    # Interpréter les résultats
    p_value = aov.anova_table['Pr > F']['Configuration']
    if p_value < 0.05:
        print(f"Différence significative détectée entre les configurations (p={p_value:.4f} < 0.05)")
    else:
        print(f"Pas de différence significative entre les configurations (p={p_value:.4f} >= 0.05)")

except Exception as e:
    print(f"Erreur lors de l'ANOVA à mesures répétées: {e}")
    print("Utilisation du test de Friedman comme alternative non paramétrique.")

# 3. Test de Friedman (alternative non paramétrique pour données appariées)
print("\n3. Test de Friedman (test non paramétrique pour données appariées):")
friedman_stat, friedman_p = stats.friedmanchisquare(*[df[col] for col in df.columns[1:7]])
print(f"Chi2={friedman_stat:.4f}, p-value={friedman_p:.4f}")

if friedman_p < 0.05:
    print(f"Différence significative détectée entre les configurations (p={friedman_p:.4f} < 0.05)")

    # Test post-hoc de Nemenyi pour identifier quelles paires de configurations diffèrent
    print("\n4. Test post-hoc (Nemenyi/Conover):")
    posthoc = sp.posthoc_nemenyi_friedman(df[df.columns[1:11]])
    print("Matrice de p-values pour les comparaisons par paires:")
    # Renommer les indices pour plus de clarté
    posthoc.index = df.columns[1:11]
    posthoc.columns = df.columns[1:11]
    print(posthoc)

    # Identifier les paires significativement différentes (p < 0.05)
    print("\nPaires de configurations significativement différentes (p < 0.05):")
    significant_pairs = []
    for i, row in enumerate(df.columns[1:11]):
        for j, col in enumerate(df.columns[1:11]):
            if i < j and posthoc.iloc[i, j] < 0.05:
                significant_pairs.append((row, col, posthoc.iloc[i, j]))

    if significant_pairs:
        for pair in significant_pairs:
            print(f"- {pair[0]} vs {pair[1]}: p={pair[2]:.4f}")
    else:
        print("Aucune paire n'est significativement différente au niveau alpha=0.05")
        # Vérifier avec un niveau alpha plus élevé
        print("\nPaires potentiellement différentes (p < 0.1):")
        trend_pairs = []
        for i, row in enumerate(df.columns[1:11]):
            for j, col in enumerate(df.columns[1:11]):
                if i < j and posthoc.iloc[i, j] < 0.1:
                    trend_pairs.append((row, col, posthoc.iloc[i, j]))

        if trend_pairs:
            for pair in trend_pairs:
                print(f"- {pair[0]} vs {pair[1]}: p={pair[2]:.4f}")
        else:
            print("Aucune paire n'est différente même au niveau alpha=0.1")
else:
    print(f"Pas de différence significative entre les configurations (p={friedman_p:.4f} >= 0.05)")

    # Examiner les différences sur la base des moyennes
    print("\nClassement des configurations basé sur les moyennes (pas de différence statistique significative):")
    sorted_means = config_means.sort_values(ascending=False)
    for i, (config, mean) in enumerate(sorted_means.items()):
        print(f"{i+1}. {config}: {mean:.2f}%")

# 4. Comparaison des configurations intraversion vs interversion
print("\n5. Analyse de l'effet intraversion vs interversion:")
# Comparer les paires de configurations qui diffèrent uniquement par intra/interversion
pairs_to_compare = [
    ('Intraversion SMOTE', 'Interversion SMOTE'),
    ('Intraversion sous-échantillonnage', 'Interversion sous-échantillonnage'),
    ('Intraversion suppression outliers', 'Interversion suppression outliers')
]

for pair in pairs_to_compare:
    stat, p = stats.wilcoxon(df[pair[0]], df[pair[1]])
    print(f"Comparaison {pair[0]} vs {pair[1]}:")
    print(f"  Test de Wilcoxon: p-value={p:.4f}")
    print(f"  Moyenne {pair[0]}: {df[pair[0]].mean():.2f}%")
    print(f"  Moyenne {pair[1]}: {df[pair[1]].mean():.2f}%")
    print(f"  Différence: {abs(df[pair[0]].mean() - df[pair[1]].mean()):.2f}%")
    if p < 0.05:
        print(f"  Différence significative (p < 0.05)")
    else:
        print(f"  Pas de différence significative (p >= 0.05)")

# 5. Analyse des différences entre méthodes de rééquilibrage
print("\n6. Analyse de l'effet des méthodes de rééquilibrage:")
# Comparer les paires de configurations qui diffèrent uniquement par la méthode de rééquilibrage
for version_type in ['Intraversion', 'Interversion']:
    print(f"\nPour {version_type}:")
    methods = [col for col in df.columns[1:11] if version_type in col]

    for i, method1 in enumerate(methods):
        for j, method2 in enumerate(methods):
            if i < j:
                stat, p = stats.wilcoxon(df[method1], df[method2])
                print(f"Comparaison {method1} vs {method2}:")
                print(f"  Test de Wilcoxon: p-value={p:.4f}")
                print(f"  Moyenne {method1}: {df[method1].mean():.2f}%")
                print(f"  Moyenne {method2}: {df[method2].mean():.2f}%")
                print(f"  Différence: {abs(df[method1].mean() - df[method2].mean()):.2f}%")
                if p < 0.05:
                    print(f"  Différence significative (p < 0.05)")
                else:
                    print(f"  Pas de différence significative (p >= 0.05)")

# Visualisations
# 1. Boxplot pour comparer les distributions
plt.figure(figsize=(14, 8))
ax = sns.boxplot(x='Configuration', y='Gmean', data=df_long)
plt.title('Distribution des scores Gmean par configuration')
plt.xticks(rotation=45, ha='right')
plt.grid(True, linestyle='--', alpha=0.7)

# Ajouter les points individuels
sns.stripplot(x='Configuration', y='Gmean', data=df_long,
              color='black', alpha=0.5, jitter=True)

plt.tight_layout()

# 2. Graphique des moyennes avec intervalles de confiance
plt.figure(figsize=(14, 8))
ax = sns.barplot(x='Configuration', y='Gmean', data=df_long, errorbar=('ci', 95), capsize=0.2)
plt.title('Scores Gmean moyens avec intervalles de confiance à 95%')
plt.xticks(rotation=45, ha='right')
plt.ylim(50, 70)  # Ajuster l'échelle pour mieux voir les différences
plt.grid(True, linestyle='--', alpha=0.7, axis='y')

# Ajouter les valeurs sur les barres
for i, p in enumerate(ax.patches):
    height = p.get_height()
    ax.text(p.get_x() + p.get_width()/2., height + 0.1,
            f'{height:.2f}%',
            ha='center', fontweight='bold')

plt.tight_layout()

# 3. Graphique de profil pour chaque système
plt.figure(figsize=(14, 8))
for i, system in enumerate(systems):
    system_data = df[df['Système'] == system]
    values = system_data[df.columns[1:11]].values[0]
    plt.plot(df.columns[1:11], values, 'o-', label=system)

plt.title('Profil de performance pour chaque système à travers les configurations')
plt.ylabel('Score Gmean (%)')
plt.xticks(rotation=45, ha='right')
plt.grid(True, linestyle='--', alpha=0.4)
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))

plt.tight_layout()
plt.show()

print("\n=== Conclusion ===")
print("L'analyse a été effectuée en utilisant des tests statistiques appropriés pour")
print("données appariées afin de déterminer s'il existe des différences significatives")
print("entre les différentes configurations testées.")

In [ ]:
!pip install scikit-posthocs

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import scikit_posthocs as sp
from statsmodels.stats.anova import AnovaRM

# Créer le DataFrame à partir des nouvelles données
systems = ['Ant', 'Camel', 'Ivy', 'Jedit', 'Log4j', 'Lucene',
           'Poi', 'Synapse', 'Velocity', 'Xalan', 'Xerces']

# Données pour chaque configuration (extraites du CV CSV.pdf)
intraversion_sous_echantillonnage_d20 = [70.23, 58.24, 70.89, 60.43, 55.25, 68.45, 66.46, 63.15, 71.33, 65.22, 51.6]
intraversion_sous_echantillonnage_d40 = [73.05, 56.49, 70.35, 60.10, 58.93, 67.09, 66.61, 60.91, 62.79, 61.20, 49.32]
interversion_sous_echantillonnage_d20 = [70.60, 60.44, 72.72, 57.05, 53.61, 56.21, 72.95, 69, 59.47, 62.61, 59.10]
interversion_sous_echantillonnage_d40 = [72.90, 58.09, 71.22, 56.25, 55.99, 54.87, 69.65, 69.59, 59.59, 60.99, 62.30]

# Créer le DataFrame
df = pd.DataFrame({
    'Système': systems,
    'Intraversion sous-échantillonnage d20': intraversion_sous_echantillonnage_d20,
    'Intraversion sous-échantillonnage d40': intraversion_sous_echantillonnage_d40,
    'Interversion sous-échantillonnage d20': interversion_sous_echantillonnage_d20,
    'Interversion sous-échantillonnage d40': interversion_sous_echantillonnage_d40,

})

print("Données utilisées pour l'analyse:")
print(df)

# Calculer les moyennes par configuration (colonne)
print("\nMoyennes par configuration:")
config_means = df.iloc[:, 1:11].mean()
print(config_means)

# Statistiques descriptives
print("\nStatistiques descriptives:")
descriptive_stats = df.iloc[:, 1:11].describe()
print(descriptive_stats)

# Réorganiser les données pour l'analyse statistique
df_long = pd.melt(df, id_vars=['Système'],
                 value_vars=df.columns[1:11],
                 var_name='Configuration', value_name='Gmean')

# 1. Vérifier la normalité des différences
print("\n1. Test de normalité (Shapiro-Wilk) pour chaque configuration:")
for config in df.columns[1:11]:
    stat, p = stats.shapiro(df[config])
    print(f"{config}: W={stat:.4f}, p-value={p:.4f}")
    if p > 0.05:
        print(f"  Les données semblent suivre une distribution normale (p > 0.05)")
    else:
        print(f"  Les données ne semblent pas suivre une distribution normale (p <= 0.05)")

# 2. ANOVA à mesures répétées (données appariées)
print("\n2. ANOVA à mesures répétées:")
try:
    # Utiliser une ANOVA à mesures répétées
    aov = AnovaRM(df_long, 'Gmean', 'Système', within=['Configuration']).fit()
    print(aov.anova_table)

    # Interpréter les résultats
    p_value = aov.anova_table['Pr > F']['Configuration']
    if p_value < 0.05:
        print(f"Différence significative détectée entre les configurations (p={p_value:.4f} < 0.05)")
    else:
        print(f"Pas de différence significative entre les configurations (p={p_value:.4f} >= 0.05)")

except Exception as e:
    print(f"Erreur lors de l'ANOVA à mesures répétées: {e}")
    print("Utilisation du test de Friedman comme alternative non paramétrique.")

# 3. Test de Friedman (alternative non paramétrique pour données appariées)
print("\n3. Test de Friedman (test non paramétrique pour données appariées):")
friedman_stat, friedman_p = stats.friedmanchisquare(*[df[col] for col in df.columns[1:7]])
print(f"Chi2={friedman_stat:.4f}, p-value={friedman_p:.4f}")

if friedman_p < 0.05:
    print(f"Différence significative détectée entre les configurations (p={friedman_p:.4f} < 0.05)")

    # Test post-hoc de Nemenyi pour identifier quelles paires de configurations diffèrent
    print("\n4. Test post-hoc (Nemenyi/Conover):")
    posthoc = sp.posthoc_nemenyi_friedman(df[df.columns[1:11]])
    print("Matrice de p-values pour les comparaisons par paires:")
    # Renommer les indices pour plus de clarté
    posthoc.index = df.columns[1:11]
    posthoc.columns = df.columns[1:11]
    print(posthoc)

    # Identifier les paires significativement différentes (p < 0.05)
    print("\nPaires de configurations significativement différentes (p < 0.05):")
    significant_pairs = []
    for i, row in enumerate(df.columns[1:11]):
        for j, col in enumerate(df.columns[1:11]):
            if i < j and posthoc.iloc[i, j] < 0.05:
                significant_pairs.append((row, col, posthoc.iloc[i, j]))

    if significant_pairs:
        for pair in significant_pairs:
            print(f"- {pair[0]} vs {pair[1]}: p={pair[2]:.4f}")
    else:
        print("Aucune paire n'est significativement différente au niveau alpha=0.05")
        # Vérifier avec un niveau alpha plus élevé
        print("\nPaires potentiellement différentes (p < 0.1):")
        trend_pairs = []
        for i, row in enumerate(df.columns[1:11]):
            for j, col in enumerate(df.columns[1:11]):
                if i < j and posthoc.iloc[i, j] < 0.1:
                    trend_pairs.append((row, col, posthoc.iloc[i, j]))

        if trend_pairs:
            for pair in trend_pairs:
                print(f"- {pair[0]} vs {pair[1]}: p={pair[2]:.4f}")
        else:
            print("Aucune paire n'est différente même au niveau alpha=0.1")
else:
    print(f"Pas de différence significative entre les configurations (p={friedman_p:.4f} >= 0.05)")

    # Examiner les différences sur la base des moyennes
    print("\nClassement des configurations basé sur les moyennes (pas de différence statistique significative):")
    sorted_means = config_means.sort_values(ascending=False)
    for i, (config, mean) in enumerate(sorted_means.items()):
        print(f"{i+1}. {config}: {mean:.2f}%")

# 4. Comparaison des configurations intraversion vs interversion
print("\n5. Analyse de l'effet intraversion vs interversion:")
# Comparer les paires de configurations qui diffèrent uniquement par intra/interversion
pairs_to_compare = [
    ('Intraversion sous-échantillonnage d20', 'Interversion sous-échantillonnage d20'),
    ('Intraversion sous-échantillonnage d40', 'Interversion sous-échantillonnage d40')
]

for pair in pairs_to_compare:
    stat, p = stats.wilcoxon(df[pair[0]], df[pair[1]])
    print(f"Comparaison {pair[0]} vs {pair[1]}:")
    print(f"  Test de Wilcoxon: p-value={p:.4f}")
    print(f"  Moyenne {pair[0]}: {df[pair[0]].mean():.2f}%")
    print(f"  Moyenne {pair[1]}: {df[pair[1]].mean():.2f}%")
    print(f"  Différence: {abs(df[pair[0]].mean() - df[pair[1]].mean()):.2f}%")
    if p < 0.05:
        print(f"  Différence significative (p < 0.05)")
    else:
        print(f"  Pas de différence significative (p >= 0.05)")

# 5. Analyse des différences entre méthodes de rééquilibrage
print("\n6. Analyse de l'effet des méthodes de rééquilibrage:")
# Comparer les paires de configurations qui diffèrent uniquement par la méthode de rééquilibrage
for version_type in ['Intraversion', 'Interversion']:
    print(f"\nPour {version_type}:")
    methods = [col for col in df.columns[1:11] if version_type in col]

    for i, method1 in enumerate(methods):
        for j, method2 in enumerate(methods):
            if i < j:
                stat, p = stats.wilcoxon(df[method1], df[method2])
                print(f"Comparaison {method1} vs {method2}:")
                print(f"  Test de Wilcoxon: p-value={p:.4f}")
                print(f"  Moyenne {method1}: {df[method1].mean():.2f}%")
                print(f"  Moyenne {method2}: {df[method2].mean():.2f}%")
                print(f"  Différence: {abs(df[method1].mean() - df[method2].mean()):.2f}%")
                if p < 0.05:
                    print(f"  Différence significative (p < 0.05)")
                else:
                    print(f"  Pas de différence significative (p >= 0.05)")

# Visualisations
# 1. Boxplot pour comparer les distributions
plt.figure(figsize=(10, 5))
ax = sns.boxplot(x='Configuration', y='Gmean', data=df_long)
plt.title('Distribution des scores Gmean par configuration')
plt.xticks(rotation=45, ha='right')
plt.grid(True, linestyle='--', alpha=0.7)

# Ajouter les points individuels
sns.stripplot(x='Configuration', y='Gmean', data=df_long,
              color='black', alpha=0.5, jitter=True)

plt.tight_layout()

# 2. Graphique des moyennes avec intervalles de confiance
plt.figure(figsize=(10, 5))
ax = sns.barplot(x='Configuration', y='Gmean', data=df_long, errorbar=('ci', 95), capsize=0.2)
plt.title('Scores Gmean moyens avec intervalles de confiance à 95%')
plt.xticks(rotation=45, ha='right')
plt.ylim(50, 70)  # Ajuster l'échelle pour mieux voir les différences
plt.grid(True, linestyle='--', alpha=0.7, axis='y')

# Ajouter les valeurs sur les barres
for i, p in enumerate(ax.patches):
    height = p.get_height()
    ax.text(p.get_x() + p.get_width()/2., height + 0.1,
            f'{height:.2f}%',
            ha='center', fontweight='bold')

plt.tight_layout()

# 3. Graphique de profil pour chaque système
plt.figure(figsize=(10, 5))
for i, system in enumerate(systems):
    system_data = df[df['Système'] == system]
    values = system_data[df.columns[1:11]].values[0]
    plt.plot(df.columns[1:11], values, 'o-', label=system)

plt.title('Profil de performance pour chaque système à travers les configurations')
plt.ylabel('Score Gmean (%)')
plt.xticks(rotation=45, ha='right')
plt.grid(True, linestyle='--', alpha=0.4)
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))

plt.tight_layout()
plt.show()

print("\n=== Conclusion ===")
print("L'analyse a été effectuée en utilisant des tests statistiques appropriés pour")
print("données appariées afin de déterminer s'il existe des différences significatives")
print("entre les différentes configurations testées.")